# 🌿 02 Training — Sistem LSTM Bonsai

| Item       | Detail                                      |
|------------|---------------------------------------------|
| **File**   | `notebooks/02_training.ipynb`                |
| **Tujuan** | Bangun arsitektur LSTM, latih model dengan data training, simpan model & histori. |
| **Input**  | `artifacts/data_train.npz`, `artifacts/label_info.json` |
| **Output** | `artifacts/model_bonsai_lstm.keras`, `artifacts/training_history.csv` |
| **Urutan** | Jalankan setelah: `01_preprocessing.ipynb` |

In [1]:
# ── VERIFIKASI LINGKUNGAN ──────────────────────────────────────────────
import sys, os

assert ".venv" in sys.executable or "venv" in sys.executable, (
    "⛔ Kernel bukan dari .venv!\n"
    "Ganti kernel ke: Python (bonsai-lstm)\n"
    f"Kernel saat ini: {sys.executable}"
)
print("✅ Kernel  :", sys.executable)
print("✅ Python  :", sys.version)
# ──────────────────────────────────────────────────────────────────────

✅ Kernel  : d:\bonsai-lstm\.venv\Scripts\python.exe
✅ Python  : 3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]


In [2]:
# ── IMPORT LIBRARY, KONSTANTA, SEED ───────────────────────────────────
import random, os
import warnings
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.utils.class_weight import compute_class_weight

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)
print(f"[SEED] Global random seed = {SEED}")

ARTIFACTS_DIR  = "../artifacts"
MODEL_PATH     = f"{ARTIFACTS_DIR}/model_bonsai_lstm.keras"
HISTORY_PATH   = f"{ARTIFACTS_DIR}/training_history.csv"

EPOCHS         = 50
BATCH_SIZE     = 32
LEARNING_RATE  = 0.001
VAL_SPLIT      = 0.10
PATIENCE       = 10

os.makedirs(ARTIFACTS_DIR, exist_ok=True)
print("✅ Konfigurasi training siap.")

[SEED] Global random seed = 42
✅ Konfigurasi training siap.


In [3]:
# ── RULE-NB-06: Validasi Artefak ───────────────────────────────────────
REQUIRED_ARTIFACTS = [
    f"{ARTIFACTS_DIR}/data_train.npz",
    f"{ARTIFACTS_DIR}/label_info.json",
]
missing = [f for f in REQUIRED_ARTIFACTS if not os.path.exists(f)]
assert not missing, (
    f"⛔ Artefak tidak ditemukan: {missing}\n"
    "Jalankan notebook sebelumnya terlebih dahulu."
)
print("✅ Semua artefak yang dibutuhkan tersedia.")

✅ Semua artefak yang dibutuhkan tersedia.


In [4]:
# ── RULE-TRAIN-02: Muat Artefak dari Preprocessing ─────────────────────
import json

train_data  = np.load(f"{ARTIFACTS_DIR}/data_train.npz")
X_train     = train_data["X_train"]
y_train     = train_data["y_train_cls"]
y_train_reg = train_data["y_train_reg"]

with open(f"{ARTIFACTS_DIR}/label_info.json") as f:
    label_info = json.load(f)

LOOK_BACK  = label_info["look_back"]
N_FEATURES = label_info["n_features"]

print(f"[LOAD] X_train : {X_train.shape}")
print(f"[LOAD] y_train : {y_train.shape}  (0={( y_train==0).sum()}, 1={(y_train==1).sum()})")

[LOAD] X_train : (3432, 24, 3)
[LOAD] y_train : (3432,)  (0=3378, 1=54)


In [5]:
# ── RULE-TRAIN-03: Penanganan Class Imbalance ───────────────────────────
classes  = np.unique(y_train)
cw_arr   = compute_class_weight("balanced", classes=classes, y=y_train)
cw_dict  = dict(zip(classes.tolist(), cw_arr.tolist()))

print(f"[CW] Class weights yang diterapkan: {cw_dict}")

[CW] Class weights yang diterapkan: {0: 0.5079928952042628, 1: 31.77777777777778}


In [6]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input
from tensorflow.keras import Model

def build_model(look_back: int, n_features: int) -> Model:
    inputs = Input(shape=(look_back, n_features))
    x = LSTM(64, return_sequences=True)(inputs)
    x = Dropout(0.2)(x)
    x = LSTM(32, return_sequences=False)(x)
    x = Dropout(0.2)(x)
    x = Dense(16, activation="relu")(x)
    
    # Output klasifikasi (binary)
    output_cls = Dense(1, activation="sigmoid", name="classification")(x)
    
    # Output regresi (soil moisture langsung dalam %)
    output_reg = Dense(1, activation="linear", name="regression")(x)
    
    model = Model(inputs=inputs, outputs=[output_cls, output_reg], name="bonsai_lstm")
    
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss={"classification": "binary_crossentropy", "regression": "mse"},
        metrics={"classification": ["accuracy", tf.keras.metrics.AUC(name="auc")], "regression": []},
        loss_weights={"classification": 1.0, "regression": 1.0},
    )
    return model

In [7]:
# ── RULE-TRAIN-05: Callbacks Wajib ──────────────────────────────────────
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, CSVLogger

callbacks = [
    EarlyStopping(
        monitor="val_loss",
        patience=PATIENCE,
        restore_best_weights=True,
        verbose=1
    ),
    ModelCheckpoint(
        filepath=MODEL_PATH,
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    ),
    CSVLogger(HISTORY_PATH, separator=",", append=False),
]
print("✅ Callbacks configured.")

✅ Callbacks configured.


In [8]:
# ── RULE-TRAIN-06: Eksekusi Pelatihan ──────────────────────────────────
history = model.fit(
    X_train, {"classification": y_train, "regression": y_train_reg},
    epochs           = EPOCHS,
    batch_size       = BATCH_SIZE,
    validation_split = VAL_SPLIT,
    callbacks        = callbacks,
    class_weight     = {"classification": cw_dict, "regression": None},
    verbose          = 2,
)
print(f"[TRAIN] Model disimpan -> {MODEL_PATH}")
print(f"[TRAIN] Histori disimpan -> {HISTORY_PATH}")

SyntaxError: unterminated string literal (detected at line 11) (1177059441.py, line 11)

In [ ]:
# ── RULE-TRAIN-07: Tampilkan Ringkasan Training ────────────────────────
hist_df    = pd.read_csv(HISTORY_PATH)
best_epoch = int(hist_df["val_loss"].idxmin()) + 1

print(f"\n[SUMMARY] Epoch terbaik    : {best_epoch} / {len(hist_df)}")
print(f"[SUMMARY] Val Loss terbaik : {hist_df['val_loss'].min():.4f}")
print(f"[SUMMARY] Val Accuracy     : {hist_df['val_accuracy'].iloc[best_epoch-1]:.4f}")
print(f"[SUMMARY] Val AUC          : {hist_df['val_auc'].iloc[best_epoch-1]:.4f}")

## 📊 Ringkasan Training

**Model yang dihasilkan:**
- `model_bonsai_lstm.keras` → Model LSTM terbaik (berdasarkan val_loss minimum)
- `training_history.csv` → Histori loss, accuracy, AUC per epoch

**Langkah selanjutnya:**
Jalankan `notebooks/03_testing.ipynb` untuk menjalankan prediksi pada data testing.